1. Raw stock price

In [11]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib
import os

# 設定字型避免中文亂碼
matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

tickers = ["0386.HK","0388.HK","0005.HK","^HSI","^GSPC" ]

os.makedirs("results", exist_ok=True)

all_results = []   # 收集所有股票的評估結果
plot_files = []    # 收集所有股票的圖表檔案路徑

for ticker in tickers:
    # 1. 下載股價
    df = yf.download(ticker, start="2023-01-01", end="2026-09-28", progress=False)
    close = df["Close"].dropna().reset_index(drop=True)

    # 存成原始收市價 CSV
    filename = f"results/Close_price_{ticker.replace('^','')}_raw.csv"
    df.to_csv(filename, index=True, encoding="utf-8-sig")

    # 2. 分割訓練集與測試集
    train = close.iloc[:746]
    test = close.iloc[746:]

    # 3. 使用 auto_arima 尋找最佳參數並訓練模型
    model = auto_arima(
        train,
        start_p=0, start_q=0,
        max_p=5, max_q=5,
        d=None,
        seasonal=False,
        stepwise=True,
        suppress_warnings=True,
        error_action='ignore'
    )
    best_order = model.order

        
    #        這裡設定 ARIMA 模型的參數 (p, d, q)。
    #        例如: 
    #        p=2 → 使用前 2 個滯後值（自迴歸項）。模型會用前 2 天的股價來解釋今天的股價。
    #        d=1 → 對資料做一次差分，讓序列平穩。把股價序列做一次差分，去掉趨勢，讓數據更平穩。
    #        q=2 → 使用前 2 個誤差項（移動平均項）。模型會考慮前 2 天的「預測誤差」來修正今天的預測。
    #        所以 (2,1,2) 的意思就是：「今天的股價 ≈ 昨天和前天的股價 + 昨天和前天的誤差 + 一點隨機噪音」。
    #        建立一個 ARIMA 模型，訓練資料是 train（通常是股價的收市價）。
    #        order=order 表示用剛剛設定的 (2,1,2) 參數。
    #        對模型進行「擬合」（fit），也就是用訓練資料去估計 ARIMA 模型的係數。完成後的 model_fit 就是一個已經訓練好的模型，可以用來做預測。

    # 4. 預測
    n_periods = len(test)
    forecast = model.predict(n_periods=n_periods)
    forecast = pd.Series(forecast, index=test.index)

    # 5. 
    mae = mean_absolute_error(test, forecast)
    rmse = np.sqrt(mean_squared_error(test, forecast))
    r2 = r2_score(test, forecast)

    # 收集結果
    all_results.append({
        "Ticker": ticker,
        "ARIMA order": best_order,
        "MAE (平均絕對誤差)": mae,
        "RMSE (均方根誤差)": rmse,
        "R² (決定係數)": r2
    })

    # 6. 繪圖並存成 PNG
    plt.figure(figsize=(14,8))
    plt.plot(train.index, train, label="訓練集（實際）", color="blue")
    plt.plot(test.index, test, label="測試集（實際）", color="green")
    plt.plot(test.index, forecast, label=f"ARIMA{best_order} 預測", color="red", linestyle="--")
    plt.axvline(x=len(train)-1, color="gray", linestyle=":", label="訓練/測試分界")
    plt.title(f"{ticker} ARIMA{best_order} 預測 vs 實際收市價\n"
              f"MAE={mae:.2f}, RMSE={rmse:.2f}, R²={r2:.2f}", fontsize=14)
    plt.xlabel("時間 (索引)")
    plt.ylabel("收市價")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plot_filename = f"results/arima_plot_{ticker.replace('^','')}.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
    plt.close()
    plot_files.append((ticker, plot_filename))

# 2) 存所有評估結果到一個 CSV
df_all = pd.DataFrame(all_results)
df_all.to_csv("results/arima_evaluation_results_all.csv", index=False, encoding="utf-8-sig")

# 3) 把所有繪圖存到一個 Excel 檔案，每個 tab 一個股票
import xlsxwriter

excel_filename = "results/arima_plots_all.xlsx"
workbook = xlsxwriter.Workbook(excel_filename)

for ticker, plot_file in plot_files:
    sheet_name = ticker.replace("^","")
    worksheet = workbook.add_worksheet(sheet_name)
    worksheet.insert_image("B2", plot_file)

# 可選：加一個 summary sheet，顯示所有評估結果
summary_ws = workbook.add_worksheet("Summary")
for col_num, col_name in enumerate(df_all.columns):
    summary_ws.write(0, col_num, col_name)
for row_num, row_data in enumerate(df_all.values, start=1):
    for col_num, value in enumerate(row_data):
        summary_ws.write(row_num, col_num, str(value))

workbook.close()


C:\Users\rc228\AppData\Local\Temp\ipykernel_8272\1385228332.py:91: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Temp\ipykernel_8272\1385228332.py:91: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Temp\ipykernel_8272\1385228332.py:91: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Temp\ipykernel_8272\1385228332.py:91: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Temp\ipykernel_8272\1385228332.py:91: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")


In [ ]:
2. Raw stock price (𝑝,𝑑,𝑞)=2,1,2

In [7]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib
import os

matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

tickers = ["0386.HK","0388.HK","0005.HK","^HSI","^GSPC" ]

os.makedirs("results", exist_ok=True)

all_results = []
plot_files = []

for ticker in tickers:
    # 1. 下載股價
    df = yf.download(ticker, start="2023-01-01", end="2026-09-28", progress=False)
    close = df["Close"].dropna().reset_index(drop=True)

    # 存成原始收市價 CSV
    filename = f"results/Close_price_{ticker.replace('^','')}_raw.csv"
    df.to_csv(filename, index=True, encoding="utf-8-sig")

    # 2. 分割訓練集與測試集
    train = close.iloc[:746]
    test = close.iloc[746:]

    # 3. 使用固定 ARIMA(2,1,2)
    model = ARIMA(train, order=(2,1,2))
    fitted = model.fit()

    # 4. 預測
    n_periods = len(test)
    forecast = fitted.forecast(steps=n_periods)
    forecast = pd.Series(forecast, index=test.index)

    # 5. 評估
    mae = mean_absolute_error(test, forecast)
    rmse = np.sqrt(mean_squared_error(test, forecast))
    r2 = r2_score(test, forecast)

    all_results.append({
        "Ticker": ticker,
        "ARIMA order": (2,1,2),
        "MAE (平均絕對誤差)": mae,
        "RMSE (均方根誤差)": rmse,
        "R² (決定係數)": r2
    })

    # 6. 繪圖
    plt.figure(figsize=(14,8))
    plt.plot(train.index, train, label="訓練集（實際）", color="blue")
    plt.plot(test.index, test, label="測試集（實際）", color="green")
    plt.plot(test.index, forecast, label="ARIMA(2,1,2) 預測", color="red", linestyle="--")
    plt.axvline(x=len(train)-1, color="gray", linestyle=":", label="訓練/測試分界")
    plt.title(f"{ticker} ARIMA(2,1,2) 預測 vs 實際收市價\n"
              f"MAE={mae:.2f}, RMSE={rmse:.2f}, R²={r2:.2f}", fontsize=14)
    plt.xlabel("時間 (索引)")
    plt.ylabel("收市價")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plot_filename = f"results/arima_plot_{ticker.replace('^','')}.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
    plt.close()
    plot_files.append((ticker, plot_filename))

# 存所有評估結果
df_all = pd.DataFrame(all_results)
df_all.to_csv("results/arima_evaluation_results_all.csv", index=False, encoding="utf-8-sig")

# 存到 Excel
import xlsxwriter
excel_filename = "results/arima_plots_all.xlsx"
workbook = xlsxwriter.Workbook(excel_filename)

for ticker, plot_file in plot_files:
    sheet_name = ticker.replace("^","")
    worksheet = workbook.add_worksheet(sheet_name)
    worksheet.insert_image("B2", plot_file)

summary_ws = workbook.add_worksheet("Summary")
for col_num, col_name in enumerate(df_all.columns):
    summary_ws.write(0, col_num, col_name)
for row_num, row_data in enumerate(df_all.values, start=1):
    for col_num, value in enumerate(row_data):
        summary_ws.write(row_num, col_num, str(value))

workbook.close()


C:\Users\rc228\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\rc228\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  start_params = self.start_params
C:\Users\rc228\AppData\Local\Temp\ipykernel_8272\639740558.py:69: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Temp\ipykernel_8272\639740558.py:69: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Temp\ipykernel_8272\639740558.py:69: UserWarning: Glyph 1

3. Raw stock price (𝑝,𝑑,𝑞)=5,2,5

In [12]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib
import os

matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False

tickers = ["0386.HK","0388.HK","0005.HK","^HSI","^GSPC" ]

os.makedirs("results", exist_ok=True)

all_results = []
plot_files = []

for ticker in tickers:
    # 1. 下載股價
    df = yf.download(ticker, start="2023-01-01", end="2026-09-28", progress=False)
    close = df["Close"].dropna().reset_index(drop=True)

    # 存成原始收市價 CSV
    filename = f"results/Close_price_{ticker.replace('^','')}_raw.csv"
    df.to_csv(filename, index=True, encoding="utf-8-sig")

    # 2. 分割訓練集與測試集
    train = close.iloc[:746]
    test = close.iloc[746:]

    # 3. 使用固定 ARIMA(5,5,5)
    model = ARIMA(train, order=(5,2,5))
    fitted = model.fit()

    # 4. 預測
    n_periods = len(test)
    forecast = fitted.forecast(steps=n_periods)
    forecast = pd.Series(forecast, index=test.index)

    # 5. 評估
    mae = mean_absolute_error(test, forecast)
    rmse = np.sqrt(mean_squared_error(test, forecast))
    r2 = r2_score(test, forecast)

    all_results.append({
        "Ticker": ticker,
        "ARIMA order": (5,2,5),
        "MAE (平均絕對誤差)": mae,
        "RMSE (均方根誤差)": rmse,
        "R² (決定係數)": r2
    })

    # 6. 繪圖
    plt.figure(figsize=(14,8))
    plt.plot(train.index, train, label="訓練集（實際）", color="blue")
    plt.plot(test.index, test, label="測試集（實際）", color="green")
    plt.plot(test.index, forecast, label="ARIMA(5,2,5) 預測", color="red", linestyle="--")
    plt.axvline(x=len(train)-1, color="gray", linestyle=":", label="訓練/測試分界")
    plt.title(f"{ticker} ARIMA(5,2,5) 預測 vs 實際收市價\n"
              f"MAE={mae:.2f}, RMSE={rmse:.2f}, R²={r2:.2f}", fontsize=14)
    plt.xlabel("時間 (索引)")
    plt.ylabel("收市價")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plot_filename = f"results/arima_plot_{ticker.replace('^','')}.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
    plt.close()
    plot_files.append((ticker, plot_filename))

# 存所有評估結果
df_all = pd.DataFrame(all_results)
df_all.to_csv("results/arima_evaluation_results_all.csv", index=False, encoding="utf-8-sig")

# 存到 Excel
import xlsxwriter
excel_filename = "results/arima_plots_all.xlsx"
workbook = xlsxwriter.Workbook(excel_filename)

for ticker, plot_file in plot_files:
    sheet_name = ticker.replace("^","")
    worksheet = workbook.add_worksheet(sheet_name)
    worksheet.insert_image("B2", plot_file)

summary_ws = workbook.add_worksheet("Summary")
for col_num, col_name in enumerate(df_all.columns):
    summary_ws.write(0, col_num, col_name)
for row_num, row_data in enumerate(df_all.values, start=1):
    for col_num, value in enumerate(row_data):
        summary_ws.write(row_num, col_num, str(value))

workbook.close()


C:\Users\rc228\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
C:\Users\rc228\AppData\Local\Temp\ipykernel_8272\3731763372.py:69: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
C:\Users\rc228\AppData\Local\Temp\ipykernel_8272\3731763372.py:69: UserWarning: Glyph 178 (\N{SUPERSCRIPT TWO}) missing from font(s) SimHei.
  plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
C:\Users\rc228\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:679: EstimationWarning: No